In [1]:
import pandas as pd
import numpy as np
import json, joblib, warnings

from sklearn.compose import TransformedTargetRegressor
from sklearn.ensemble import RandomForestRegressor, HistGradientBoostingRegressor
from sklearn.model_selection import cross_validate, train_test_split, RandomizedSearchCV
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

warnings.filterwarnings('ignore', category=UserWarning)

from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
from data_pipeline import modeling_pipeline
data = pd.read_csv('/content/drive/MyDrive/Portnet Imputation Prediction/cleaned_data.csv')

# HuggingFace Config

In [ ]:
from huggingface_hub import notebook_login, HfApi, create_repo

notebook_login()

In [5]:
repo_id = "Meliodas-10/portnet-model"
create_repo(repo_id=repo_id, repo_type="model", exist_ok=True)

RepoUrl('https://huggingface.co/Meliodas-10/portnet-model', endpoint='https://huggingface.co', repo_type='model', repo_id='Meliodas-10/portnet-model')

In [8]:
data.describe()

,ANNEE,MOIS,FRET,FOB,ASSURANCE,FRAIS_ACCESSOIRE,CODE_SH,QUANTITE_DOMICILE,QTE_IMPUTE,DELAI_ENREGISTREMENT,TRIMESTRE,MOIS_SIN,MOIS_COS,A_UNE_ASSURANCE,A_FRAIS_ACCESSOIRE
count,3.332655e+06,3.332655e+06,3.332655e+06,3.332655e+06,3.332653e+06,3.332655e+06,3.332655e+06,3.332655e+06,3.332655e+06,3.332655e+06,3.332655e+06,3.332655e+06,3.332655e+06,3.332655e+06,3.332655e+06
mean,2.019173e+03,6.601376e+00,1.430064e+04,3.141672e+05,7.576115e+01,1.689095e+00,6.074027e+09,1.319580e+05,5.819991e+04,9.251401e+00,2.530407e+00,-1.503772e-03,2.113357e-02,1.634853e-02,1.529111e-03
std,2.467559e+00,3.485217e+00,2.511013e+05,3.656671e+06,5.027887e+03,3.002232e+02,2.584613e+09,9.048811e+05,5.452835e+05,4.273548e+01,1.136582e+00,7.038271e-01,7.100555e-01,1.268119e-01,3.907395e-02
min,2.014000e+03,1.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,1.012100e+08,1.000000e-03,1.000000e-03,0.000000e+00,1.000000e+00,-1.000000e+00,-1.000000e+00,0.000000e+00,0.000000e+00
25%,2.017000e+03,4.000000e+00,0.000000e+00,4.410000e+03,0.000000e+00,0.000000e+00,3.917400e+09,2.094000e+02,1.240000e+02,0.000000e+00,2.000000e+00,-5.000000e-01,-5.000000e-01,0.000000e+00,0.000000e+00
50%,2.019000e+03,7.000000e+00,3.393200e+01,1.554140e+04,0.000000e+00,0.000000e+00,6.804210e+09,4.350000e+03,2.703700e+03,0.000000e+00,3.000000e+00,-2.449294e-16,6.123234e-17,0.000000e+00,0.000000e+00
75%,2.021000e+03,1.000000e+01,1.342000e+03,4.499881e+04,0.000000e+00,0.000000e+00,8.481809e+09,2.208000e+04,1.802400e+04,0.000000e+00,4.000000e+00,5.000000e-01,8.660254e-01,0.000000e+00,0.000000e+00
max,2.024000e+03,1.200000e+01,1.495315e+07,2.038911e+08,3.048725e+06,2.850000e+05,9.706000e+09,2.000000e+07,2.000000e+07,3.650000e+02,4.000000e+00,1.000000e+00,1.000000e+00,1.000000e+00,1.000000e+00


# Dataset Config

In [12]:
X = data.drop(columns=['QTE_IMPUTE'])
y = data['QTE_IMPUTE']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.24)

print(f'Taille du train set : {X_train.shape[0]} lignes')
print(f'Taille du test set final : {X_test.shape[0]} lignes')

X_train_sample = X_train.sample(n=200000, random_state=42)
y_train_sample = y_train.loc[X_train_sample.index]

Taille du train set : 2532817 lignes
Taille du test set final : 799838 lignes


# 1- Algorithm Selection and Training

## 1.1- Dummy Model

In [13]:
from sklearn.dummy import DummyRegressor

dummy_model = TransformedTargetRegressor(
    regressor=DummyRegressor(strategy='mean'),
    func=np.log1p,
    inverse_func=np.expm1
)

dummy_model.fit(X_train_sample, y_train_sample)

y_pred_baseline = dummy_model.predict(X_test)

mae_base = mean_absolute_error(y_test, y_pred_baseline)
rmse_base = np.sqrt(mean_squared_error(y_test, y_pred_baseline))
r2_base = r2_score(y_test, y_pred_baseline)

print("--- BASELINE Score ---")
print(f"MAE  : {mae_base:.2f}")
print(f"RMSE : {rmse_base:.2f}")
print(f"R²   : {r2_base:.4f}")

--- BASELINE Score ---
MAE  : 57622.38
RMSE : 545015.58
R²   : -0.0109


## 1.2- Models CV Loop

In [14]:
algorithmes = {
    "Random Forest (Bagging)": RandomForestRegressor(n_estimators=50, random_state=42, n_jobs=-1),
    "HistGradientBoosting (Boosting)": HistGradientBoostingRegressor(random_state=42)
}

In [15]:
test_slice = X_train.head(5)
target_slice = y_train.head(5)

print("Testing pipeline on a small slice...")
modele_complet = TransformedTargetRegressor(
    regressor=modeling_pipeline.set_params(algorithme=algorithmes["Random Forest (Bagging)"]),
    func=np.log1p,
    inverse_func=np.expm1
)
modele_complet.fit(test_slice, target_slice)
print("\n----- Sanity check passed successfully! The pipeline fits without errors -----")

Testing pipeline on a small slice...
[_logarithmic_transform] exécuté en 0.001s | Lignes restantes: 5
[_impute_and_scale] exécuté en 0.003s | Lignes restantes: 5
[_target_encode] exécuté en 0.001s | Lignes restantes: 5

----- Sanity check passed successfully! The pipeline fits without errors -----


In [16]:
for nom, algo in algorithmes.items():
    print(f"\n--- Entraînement de {nom} en cours (CV = 5) ---")
    modeling_pipeline.set_params(algorithme=algo)

    modele_complet = TransformedTargetRegressor(
        regressor=modeling_pipeline,
        func=np.log1p,
        inverse_func=np.expm1
    )

    scores = cross_validate(
        modele_complet, 
        X_train_sample, 
        y_train_sample, 
        cv=5, 
        scoring=('neg_mean_absolute_error', 'r2'),
        n_jobs=1
    )
 
    mae_moyen = -scores['test_neg_mean_absolute_error'].mean()
    r2_moyen = scores['test_r2'].mean()
    
    print(f"[{nom}] R² Moyen  : {r2_moyen:.4f}")
    print(f"[{nom}] MAE Moyen : {mae_moyen:.2f} unités")

## Results:
Random Forest (Bagging):
* R² Moyen  : 0.8686
* MAE Moyen : 14569.53 unités


HistGradientBoosting (Boosting):
* R² Moyen  : 0.7668
* MAE Moyen : 21491.05 unités

### --> Decision : Random Forest

# 2- Hyperparameter Tuning

In [17]:
modeling_pipeline.set_params(algorithme=RandomForestRegressor(n_estimators=50, random_state=42, n_jobs=-1))

model = TransformedTargetRegressor(
    regressor=modeling_pipeline,
    func=np.log1p,
    inverse_func=np.expm1
)

param_distributions = {
    'regressor__algorithme__n_estimators': [100, 200, 300],
    'regressor__algorithme__max_depth': [15, 25, None],
    'regressor__algorithme__min_samples_split': [2, 5, 10],
    'regressor__algorithme__min_samples_leaf': [1, 2, 4]
}

random_search = RandomizedSearchCV(
    estimator=model,
    param_distributions=param_distributions,
    n_iter=5,
    cv=3,
    scoring='r2',
    random_state=42,
    n_jobs=1
)

In [18]:
print("--- Début de l'optimisation des hyperparamètres ---")
random_search.fit(X_train_sample, y_train_sample)

print(f"\nMeilleurs paramètres trouvés : {random_search.best_params_}")
print(f"Meilleur score R² (CV) : {random_search.best_score_:.4f}")

best_params = {k.replace('regressor__', ''): v for k, v in random_search.best_params_.items()}
modeling_pipeline.set_params(**best_params)

with open("best_params.json", "w") as f:
    json.dump(best_params, f)

api = HfApi()
api.upload_file(
    path_or_fileobj="best_params.json",
    path_in_repo="best_params.json",
    repo_id=repo_id,
    repo_type="model"
)

print("\nParamètres sauvegardés avec succès sur Hugging Face !")

# 3- Model Refitting

In [19]:
for param, value in modeling_pipeline.named_steps['algorithme'].get_params().items():
    print(f"{param}: {value}")

bootstrap: True
ccp_alpha: 0.0
criterion: squared_error
max_depth: 15
max_features: 1.0
max_leaf_nodes: None
max_samples: None
min_impurity_decrease: 0.0
min_samples_leaf: 4
min_samples_split: 5
min_weight_fraction_leaf: 0.0
monotonic_cst: None
n_estimators: 200
n_jobs: -1
oob_score: False
random_state: 42
verbose: 0
warm_start: False


In [20]:
full_model = TransformedTargetRegressor(
    regressor=modeling_pipeline,
    func=np.log1p,
    inverse_func=np.expm1
)

print("\n--- Entraînement du modèle final sur l'intégralité du dataset ---")
full_model.fit(X_train, y_train)


--- Entraînement du modèle final sur l'intégralité du dataset ---
[_logarithmic_transform] exécuté en 0.063s | Lignes restantes: 2532817
[_impute_and_scale] exécuté en 0.154s | Lignes restantes: 2532817
[_target_encode] exécuté en 0.301s | Lignes restantes: 2532817


TransformedTargetRegressor(func=<ufunc 'log1p'>, inverse_func=<ufunc 'expm1'>,
                           regressor=Pipeline(steps=[('preprocessor',
                                                      PortNetDataPreprocessing()),
                                                     ('algorithme',
                                                      RandomForestRegressor(max_depth=15,
                                                                            min_samples_leaf=4,
                                                                            min_samples_split=5,
                                                                            n_estimators=200,
                                                                            n_jobs=-1,
                                                                            random_state=42))]))

# 4- Model Saving

In [21]:
joblib.dump(full_model, "portnet_model.pkl")
api = HfApi()

api.upload_file(
    path_or_fileobj="portnet_model.pkl",
    path_in_repo="portnet_model.pkl",
    repo_id=repo_id,
    repo_type="model"
)
print("modèle sauvegardés avec succès sur Hugging Face !")

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  /content/portnet_model.pkl  :   0%|          |  992kB /  369MB            

modèle sauvegardés avec succès sur Hugging Face !


# END